# Demostración funcional — NYC Taxi Big Data (≤ 3 min)

Este notebook corre en el contenedor `spark-client` (driver) y se conecta al clúster Spark standalone (`spark://spark-master:7077`, 2 workers). Recorre las capas **Bronze → Silver → Gold → Modelo → Servicio**.

In [1]:
import sys; sys.path.insert(0, '/opt/project/src')
from pyspark.sql import functions as F
from nyc_taxi.config import get_settings
from nyc_taxi.utils import build_spark
s = get_settings()
spark = build_spark(s, 'demo-notebook', {'spark.ui.showConsoleProgress': 'false'})
print('Spark', spark.version, '| master =', spark.sparkContext.master, '| app =', spark.sparkContext.applicationId)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 21:51:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/12 21:51:56 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Spark 3.5.5 | master = spark://spark-master:7077 | app = app-20261112215156-0008


## 1. Ingesta y almacenamiento: capa Bronze particionada por año/mes

In [2]:
bronze = spark.read.parquet(s.bronze_trips)
bronze.groupBy('year', 'month').count().orderBy('month').show(12)
print('Filas en Bronze:', f'{bronze.count():,}')

+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2025|    1|3475226|
|2025|    2|3577543|
|2025|    3|4145257|
|2025|    4|3970553|
|2025|    5|4591845|
|2025|    6|4322960|
|2025|    7|3898963|
|2025|    8|3574091|
|2025|    9|4251015|
|2025|   10|4428699|
|2025|   11|4181444|
|2025|   12|4305006|
+----+-----+-------+



Filas en Bronze: 48,722,602


## 2. Procesamiento: Silver limpio, enriquecido y cuarentena auditable

In [3]:
silver = spark.read.parquet(s.silver_trips)
quarantine = spark.read.parquet(s.silver_quarantine)
print(f'Silver: {silver.count():,} | Cuarentena: {quarantine.count():,}')
quarantine.groupBy('dq_first_failure').count().orderBy(F.desc('count')).show(truncate=False)
silver.select('pickup_datetime','pu_zone','do_zone','trip_distance','duration_min','avg_speed_mph','time_band','centroid_dist_mi','total_amount').show(5, truncate=False)

Silver: 43,688,645 | Cuarentena: 5,033,957


+--------------------+-------+
|dq_first_failure    |count  |
+--------------------+-------+
|tarifa_valida       |2579138|
|distancia_valida    |1021306|
|duracion_valida     |563390 |
|fin_posterior_inicio|546304 |
|zonas_validas       |320434 |
|velocidad_plausible |3312   |
|pasajeros_validos   |44     |
|dentro_del_periodo  |29     |
+--------------------+-------+



+-------------------+------------+--------------+-------------+------------+-------------+-----------------+----------------+------------+
|pickup_datetime    |pu_zone     |do_zone       |trip_distance|duration_min|avg_speed_mph|time_band        |centroid_dist_mi|total_amount|
+-------------------+------------+--------------+-------------+------------+-------------+-----------------+----------------+------------+
|2025-07-08 13:07:14|Central Park|Midtown North |0.1          |1.8         |3.33         |Día (6-15)       |1.396           |8.45        |
|2025-07-23 22:39:00|East Chelsea|Clinton East  |1.85         |9.0         |12.33        |Noche (20-23)    |1.091           |21.5        |
|2025-07-24 08:45:59|East Chelsea|Midtown Center|0.1          |21.47       |0.28         |Pico mañana (6-9)|1.341           |29.85       |
|2025-07-20 12:40:15|East Chelsea|Clinton East  |1.85         |10.1        |10.99        |Día (6-15)       |1.091           |18.05       |
|2025-07-27 22:00:28|East C

## 3. Analítica distribuida con Spark SQL (consulta ad-hoc)

In [4]:
silver.createOrReplaceTempView('trips')
spark.sql('''
  SELECT pu_zone, COUNT(*) AS viajes, ROUND(AVG(avg_speed_mph),1) AS mph
  FROM trips WHERE NOT is_weekend AND pickup_hour BETWEEN 17 AND 19
  GROUP BY pu_zone ORDER BY viajes DESC LIMIT 10''').show(truncate=False)

26/11/12 21:52:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------------------+------+----+
|pu_zone                     |viajes|mph |
+----------------------------+------+----+
|Midtown Center              |398171|7.1 |
|Upper East Side South       |329508|8.4 |
|Midtown East                |283641|7.9 |
|Upper East Side North       |268768|8.8 |
|JFK Airport                 |249533|20.2|
|Midtown North               |211912|7.5 |
|Times Sq/Theatre District   |205431|7.3 |
|Union Sq                    |202595|7.9 |
|Murray Hill                 |196947|8.1 |
|Penn Station/Madison Sq West|180152|7.7 |
+----------------------------+------+----+



## 4. Modelo MLlib persistido: predicción de duración para un viaje nuevo

In [5]:
from pyspark.ml import PipelineModel
from nyc_taxi.model import load_dataset
gbt = PipelineModel.load(str(s.models_dir / 'gbt_duration_min'))
ejemplo = load_dataset(spark, s).limit(5)
gbt.transform(ejemplo).select('pu_borough','do_borough','trip_distance','pickup_hour','duration_min', F.round('prediction', 1).alias('pred_min')).show()

+----------+----------+-------------+-----------+------------+--------+
|pu_borough|do_borough|trip_distance|pickup_hour|duration_min|pred_min|
+----------+----------+-------------+-----------+------------+--------+
| Manhattan| Manhattan|          0.1|         20|       11.42|     4.6|
|    Queens|    Queens|          6.9|          6|       24.38|    18.3|
| Manhattan| Manhattan|          0.2|         17|         2.0|     4.2|
| Manhattan| Manhattan|          7.0|         23|       25.12|    23.6|
|    Queens|     Bronx|          7.1|         10|       23.25|    25.7|
+----------+----------+-------------+-----------+------------+--------+



## 5. Capa de servicio: DuckDB sobre las tablas Gold (sin Spark)

In [6]:
import duckdb
con = duckdb.connect('/opt/project/outputs/serving/nyc_taxi_gold.duckdb', read_only=True)
con.sql("SELECT borough, SUM(trips) AS viajes, ROUND(SUM(revenue_usd)/1e6,1) AS ingresos_musd "
        "FROM mensual_borough GROUP BY borough ORDER BY viajes DESC").show()

┌───────────────┬──────────┬───────────────┐
│    borough    │  viajes  │ ingresos_musd │
│    varchar    │  int128  │    double     │
├───────────────┼──────────┼───────────────┤
│ Manhattan     │ 37886268 │         923.5 │
│ Queens        │  3997103 │         262.1 │
│ Brooklyn      │  1464112 │          44.2 │
│ Bronx         │   337263 │          10.7 │
│ Staten Island │     3589 │           0.2 │
│ EWR           │      310 │           0.0 │
└───────────────┴──────────┴───────────────┘



In [7]:
spark.stop()